# TableWatch - run on a video

Give it a video from a **fixed camera** (phone on a tripod or shelf, CCTV). It will:

1. find the tables,
2. decide for each one, over time: **occupied**, **clean** or **dirty**,
3. raise an alert when a table has been dirty for too long,
4. optionally score itself against your hand-written ground truth.

Nothing is trained: people and tables are found with an off-the-shelf COCO detector, and
"dirty" means "this empty table no longer looks like it did when it was clean".

**Runtime -> Change runtime type -> T4 GPU** makes it much faster (CPU works for short clips).

In [ ]:
REPO_URL = "https://github.com/kvarun10/Smart-Restaurant-Monitoring-System.git"

import os, sys, json
REPO = "/content/tablewatch"
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
!pip -q install ultralytics
print("code version:", os.popen(f"git -C {REPO} log -1 --format=%h_%s").read().strip())

## 1. Video

Upload a file, or set `VIDEO` to a path in your Google Drive for long recordings.
The tables should be **clean and empty the first time each one is visible without people** -
that first view becomes the table's "clean" reference.

In [ ]:
from google.colab import files
VIDEO = None   # e.g. "/content/drive/MyDrive/tablewatch/cafe_cam1.mp4" after mounting Drive
if VIDEO is None:
    VIDEO = next(iter(files.upload()))
RUN_NAME = os.path.splitext(os.path.basename(VIDEO))[0]
OUT = f"/content/report/runs/{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)

## 2. Find the tables

In [ ]:
import cv2
from IPython.display import Image, display
from ultralytics import YOLO
from tablewatch.pipeline import discover_zones

MODEL = "yolo11s.pt"
zones = discover_zones(YOLO(MODEL), VIDEO)
ZONES = {z.id: [round(v) for v in z.box] for z in zones}

def preview(zone_dict):
    cap = cv2.VideoCapture(VIDEO); ok, frame = cap.read(); cap.release()
    for name, (x1, y1, x2, y2) in zone_dict.items():
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 200, 0), 2)
        cv2.putText(frame, name, (int(x1) + 4, int(y1) + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 200, 0), 2)
    cv2.imwrite(f"{OUT}/zones_preview.jpg", frame)
    display(Image(f"{OUT}/zones_preview.jpg", width=900))

preview(ZONES)
print(json.dumps(ZONES))

**Check the picture.** Each box should cover one table top. If a table is missing or a box is
wrong, fix `ZONES` here (`[x1, y1, x2, y2]` in pixels; x grows to the right, y grows downwards)
and re-run this cell. Keep the IDs - your ground-truth file refers to them.

In [ ]:
# ZONES["T5"] = [600, 250, 835, 300]   # add or correct a table
# del ZONES["T3"]                       # remove a false one
preview(ZONES)

## 3. Run

In [ ]:
from tablewatch.pipeline import run_video
from tablewatch.monitor import MonitorConfig
from tablewatch.tracker import TrackerConfig

SETTINGS = dict(
    dirty_threshold=0.10,   # share of a table that must look different from its clean reference
    dirty_alert_s=120,      # alert after a table has been dirty this long (5 min in a real cafe)
    stride=3,               # analyse every 3rd frame
    max_seconds=None,       # set e.g. 120 for a quick first try
)
cfg = MonitorConfig(dirty_threshold=SETTINGS["dirty_threshold"],
                    tracker=TrackerConfig(dirty_alert_s=SETTINGS["dirty_alert_s"]))
summary = run_video(VIDEO, OUT, zones=ZONES, model=MODEL, config=cfg,
                    stride=SETTINGS["stride"], max_seconds=SETTINGS["max_seconds"])
json.dump({"video": os.path.basename(VIDEO), "model": MODEL, **SETTINGS}, open(f"{OUT}/settings.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
import pandas as pd
events = json.load(open(f"{OUT}/events.json"))
pd.DataFrame(events)

In [ ]:
# Watch the result (re-encoded so the browser can play it). For long videos, download it instead.
!ffmpeg -y -loglevel error -i {OUT}/annotated.mp4 -vcodec libx264 -pix_fmt yuv420p -vf scale=960:-2 /content/annotated_h264.mp4
from IPython.display import Video
Video("/content/annotated_h264.mp4", embed=True, width=900)

## 4. Score it against ground truth (optional, but this is what makes a result)

Watch the original video and write down what really happened to each table, using the table
IDs from the picture above. Save as a CSV like `data/annotations_example.csv` in the repo:

```
table,start_s,end_s,state
T1,0,95,occupied
T1,95,260,unoccupied_dirty
T1,260,600,unoccupied_clean
```

Write the ground truth **before** looking at what the system said.

In [ ]:
from tablewatch.evaluate import evaluate
print("Upload the ground-truth CSV for this video")
ANNOTATIONS = next(iter(files.upload()))
result = evaluate(ANNOTATIONS, events, summary["duration_s"])
json.dump(result, open(f"{OUT}/evaluation.json", "w"), indent=2)
os.replace(ANNOTATIONS, f"{OUT}/ground_truth.csv")

print("state accuracy over time:", result["time"]["accuracy"], "on", result["time"]["annotated_seconds"], "table-seconds")
display(pd.DataFrame(result["time"]["per_state"]).T)
display(pd.DataFrame(result["time"]["seconds"]).T.rename_axis("true state / predicted (seconds)"))
pd.Series(result["dirty_episodes"])

## 5. Save the evidence

Unzip into `report/runs/` in the repo on your PC (without the big video) and commit.

In [ ]:
import shutil
shutil.make_archive(f"/content/{RUN_NAME}_evidence", "zip", OUT)
files.download(f"/content/{RUN_NAME}_evidence.zip")